# Figure 1 — LinkD-Bind framework and affinity benchmarking

**Caption (abridged):** The LinkD framework integrates proteome-wide affinity prediction, selectivity scoring, phenotypic validation, and clinical evidence. LinkD-Bind outperforms established drug-target affinity predictors on BindingDB, Davis, and KIBA under random, cold-drug, and cold-protein splits.


## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

- `source_data/TableS2_Benchmarking_LinkD.xlsx`
- `source_data/Ensemble_result_concat_regression.csv`

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Panel a — Framework schematic
*Not regenerated from tabular data.*

In [ ]:
illustrate.show_panel('fig1_a', title='Panel a')

## Panel b — Rank heatmap across models and splits
Each cell is the rank of a model in a split (1 = best). Built from TableS2 test-set Pearson correlations.

In [ ]:

df = io.read_table_s2()
test = df[df["Dataset"] == "Test"].copy()
# Average Pearson across BindDB/Davis/Kiba within each Mode; then rank models (higher Pearson = better = rank 1)
pivot = test.pivot_table(index="Model", columns="Mode", values="Pearson", aggfunc="mean")
# Keep models with all three modes when possible; drop pure-missing
pivot = pivot.dropna(how="all")
ranks = pivot.rank(ascending=False, method="min")
# Manuscript cites ~13 models; keep models present in random split
keep = ranks.dropna(subset=["random"]).index.tolist()
ranks = ranks.loc[keep]
# Order models by mean rank
order = ranks.mean(axis=1).sort_values().index
ranks = ranks.loc[order]

fig, ax = plt.subplots(figsize=(4.2, 5.2))
im = ax.imshow(ranks.values, cmap="viridis_r", aspect="auto", vmin=1, vmax=max(13, ranks.max().max()))
ax.set_xticks(range(len(ranks.columns)))
ax.set_xticklabels(ranks.columns, rotation=30, ha="right")
ax.set_yticks(range(len(ranks.index)))
ax.set_yticklabels(ranks.index)
ax.set_title("Fig 1b — mean Pearson rank (1=best)")
cbar = fig.colorbar(im, ax=ax, fraction=0.046)
cbar.set_label("Rank")
for i in range(ranks.shape[0]):
    for j in range(ranks.shape[1]):
        v = ranks.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f"{int(v)}", ha="center", va="center", color="white" if v > ranks.values.max()/2 else "black", fontsize=6)
fig.tight_layout()
out = style.save_panel(fig, "fig1_b_rank_heatmap", ranks.reset_index())
plt.show()
print(out)
# Claim: LinkD should be among top ranks
if "LinkD" in ranks.index:
    print("LinkD ranks:", ranks.loc["LinkD"].to_dict())


## Panel c — Mean test Pearson: LinkD vs ablations and deep baselines

In [ ]:

df = io.read_table_s2()
test = df[df["Dataset"] == "Test"].copy()
focus = ["LinkD", "Diffusion", "MLP", "DeepDTA", "DeepPurpose", "GraphDTA"]
sub = test[test["Model"].isin(focus)]
mean_r = sub.groupby(["Model", "Mode"])["Pearson"].mean().reset_index()
wide = mean_r.pivot(index="Mode", columns="Model", values="Pearson")
# order modes
mode_order = [m for m in ["random", "cold_protein", "cold_drug"] if m in wide.index]
wide = wide.loc[mode_order]

fig, ax = plt.subplots(figsize=(5.5, 3.2))
x = np.arange(len(wide.index))
for model in [m for m in focus if m in wide.columns]:
    ax.plot(x, wide[model].values, marker="o", label=model, linewidth=1.2, markersize=4)
ax.set_xticks(x)
ax.set_xticklabels(wide.index)
ax.set_ylabel("Mean test Pearson r")
ax.set_title("Fig 1c — LinkD vs ablations / deep baselines")
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
out = style.save_panel(fig, "fig1_c_ablation_lines", wide.reset_index())
plt.show()
print(out)
